# PolicyPilot: long context vs RAG on a free Colab T4

Same pre-registered grid as the Kaggle notebook (`docs/LONG_CONTEXT_PLAN.md`): Qwen2.5-7B-Instruct
(4-bit AWQ, 32K window) on vLLM with prefix caching, on **one** free T4. No paid API, no keys.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

**Free Colab disconnects** (idle tabs, or after a few hours). Every finished requirement is saved to
your Google Drive (`MyDrive/policypilot_long_context/long_context_runs.jsonl`), so after a disconnect
just reconnect and **Run all** again: finished rows are skipped. Expect several sessions for the
whole grid (about 1,000 model calls; time per call not yet measured on a single T4).

In [ ]:
# 1. Save progress to Google Drive, so a disconnect loses nothing
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/policypilot_long_context"
os.makedirs(OUT, exist_ok=True)
RUNS = f"{OUT}/long_context_runs.jsonl"
print("Checkpoint:", RUNS, "(rows so far:", sum(1 for _ in open(RUNS)) if os.path.exists(RUNS) else 0, ")")

In [ ]:
# 2. Get the code
import subprocess
if not os.path.exists("/content/policypilot"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "long-context",
                    "https://github.com/vik-bit/policypilot.git", "/content/policypilot"], check=True)
os.chdir("/content/policypilot")
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Install (a few minutes). vLLM brings its own torch; PolicyPilot is installed without its
#    web/agent extras, which long-context mode does not import.
!pip install -q "vllm==0.6.6.post1"
!pip install -q --no-deps -e .
!pip install -q "fastembed>=0.4" "tokenizers>=0.20" pypdf

In [ ]:
# 4. Start vLLM in the background on the single T4 and wait until it answers
import time, urllib.request
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

def start(extra):
    cmd = ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--quantization", "awq",
           "--dtype", "half", "--max-model-len", "32768", "--gpu-memory-utilization", "0.92",
           "--enable-prefix-caching", "--port", "8000", *extra]
    return subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

def wait(proc, minutes=20):
    for _ in range(minutes * 12):
        if proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            return True
        except OSError:
            time.sleep(5)
    return False

# Ask vLLM to report cached prompt tokens; a build without the flag is started without it.
server = start(["--enable-prompt-tokens-details"])
if not wait(server):
    server.kill(); server = start([])
    assert wait(server), open("/content/vllm.log").read()[-3000:]
print("vLLM is up")

In [ ]:
# 5. Exact prompt sizes with the Qwen tokenizer (no model call)
!python -m policypilot.longctx tokens --distractors 0 50 133

In [ ]:
# 6. Smoke check on 5 requirements of the development set (not the held-out set)
!python -m policypilot.longctx --model {MODEL} eval --grid smoke --out /content/smoke_runs.jsonl
!tail -n 2 /content/smoke_runs.jsonl | cut -c 1-600

In [ ]:
# 7. The pre-registered grid on the held-out `real` set. After a disconnect, Run all again.
!python -m policypilot.longctx --model {MODEL} eval --grid main --out "{RUNS}"

In [ ]:
# 8. Tables, intervals and paired tests (no model call); copies land in your Drive folder
import shutil
shutil.copy(RUNS, "data/eval/long_context_runs.jsonl")
!python -m policypilot.longctx report
shutil.copy("docs/LONG_CONTEXT_RESULTS.md", f"{OUT}/LONG_CONTEXT_RESULTS.md")
shutil.copy("data/eval/long_context_results.json", f"{OUT}/long_context_results.json")